# 5. The comparison

Notebooks 01 to 04 each saved a metrics row and, for `toy1d`, the curves needed to
draw a figure. This notebook loads them, produces the table and the three figures
the report asks for, and lists the questions you need to answer.

There is nothing to implement here. If a cell complains that a file is missing,
run the notebook it names first.

About 1.5 hours, most of it writing rather than running.

In [ ]:
import sys

sys.path.insert(0, "..")

import numpy as np

from bdl.data import load_track, make_toy1d, split_fingerprint, toy1d_grid
from bdl.metrics import results_table
from bdl.plots import plot_aleatoric_recovery, plot_bands, plot_reliability, plot_uncertainty_vs_x
from bdl.store import load_runs, run_dir

%matplotlib inline

TRACK = "A"  # <-- the same track you used in notebooks 01 to 04

ds = make_toy1d()
grid = toy1d_grid()
toy = load_runs("toy1d")
track = load_runs(TRACK)
print("toy1d :", list(toy))
print(TRACK, ":", list(track))

expected = ["deterministic", "mc_dropout", "ensemble", "bbb"]
notebooks = dict(zip(expected, ["01", "02", "03", "04"]))
missing = [m for m in expected if m not in toy or m not in track]
assert not missing, "no results for " + ", ".join(
    f"{m} (run notebook {notebooks[m]})" for m in missing
)

## 5.1 The two tables

Both tables go in the report. `toy1d` first, because it is the one where the truth
is known and where a broken implementation is obvious.

In [ ]:
print("toy1d      fingerprint", split_fingerprint(ds))
print(results_table({name: r["metrics"] for name, r in toy.items()}))
print()
ds_track = load_track(TRACK)
print(f"track {TRACK}    fingerprint", split_fingerprint(ds_track), " ", ds_track.name)
print(results_table({name: r["metrics"] for name, r in track.items()}))

## 5.2 Figure 1: the predictive bands

Four panels, one per method. The inner band is aleatoric, the outer band is the
total, so the space between them is the epistemic part. Read it in three places:
on the data, inside the gap, and beyond the training range.

In [ ]:
panels = {
    name: {
        "mean": r["arrays"]["mean"],
        "sd_aleatoric": r["arrays"]["sd_aleatoric"],
        "sd_total": r["arrays"]["sd_total"],
    }
    for name, r in toy.items()
}
plot_bands(panels, grid, ds, run_dir("toy1d") / "predictive_bands.png");

## 5.3 Figure 2: the reliability diagram

Nominal credible level against the coverage actually observed on the test set. The
diagonal is honesty. Below it the intervals are too tight, above it too wide.

In [ ]:
curves = {name: (r["arrays"]["levels"], r["arrays"]["coverage"]) for name, r in toy.items()}
plot_reliability(curves, run_dir("toy1d") / "reliability.png");

## 5.4 Figure 3: does uncertainty grow where the data stops?

The epistemic standard deviation against `x`, with the regions that have no
training data shaded. Compare shapes, not heights: the methods are not on a common
scale, and a method whose curve is flat has told you nothing regardless of where it
sits.

In [ ]:
epi = {name: r["arrays"]["epistemic_std"] for name, r in toy.items()}
# The deterministic baseline is exactly zero everywhere, which cannot be drawn on
# a log axis, so it is nudged to a floor to keep it visible as a flat line.
epi["deterministic"] = np.maximum(np.asarray(epi["deterministic"], dtype=float), 1e-6)
plot_uncertainty_vs_x(epi, grid, ds, run_dir("toy1d") / "uncertainty_vs_x.png");

## 5.5 A fourth figure worth having

`toy1d` is the only dataset here whose true observation noise is known, so the
aleatoric estimate can be checked against it rather than trusted. This is usually
the most interesting figure in the set.

In [ ]:
ale = {name: r["arrays"]["sd_aleatoric"] for name, r in toy.items()}
plot_aleatoric_recovery(ale, grid, run_dir("toy1d") / "aleatoric_recovery.png");

## 5.6 Questions to answer in the report

These four show up in every correct run. Answer them from your own numbers, and if
your numbers disagree with what is described here, say so and investigate: that is
a result, not a mistake.

**1. The baseline has an OOD AUROC of exactly 0.500 and a shifted NLL in the
hundreds. Why are those the same fact?**

**2. The ensemble usually has the best RMSE. Look at its `coverage@95` in the same
row.** Best fit and worst honesty can occur together. Two contributions are
measurable: the population variance over `M` members underestimates the spread by
a factor `1 - 1/M`, and each member's `sigma(x)` was fitted to its own training
residuals.

**3. Bayes by Backprop fits worst and is often the best calibrated.** On some
tracks it is also the only method still usable on the shifted set. Explain the
trade, and use section 4.7: its aleatoric estimate is inflated by exactly what its
epistemic term is missing, so the total can be right while the split is not.

**4. The winner is not the same on every track, and not the same in every column.**
Say which method you would use for a point prediction, which for a decision that
depends on the error bar, and why those are different answers.

Two habits worth keeping while you write. Do not compare epistemic magnitudes
across methods, only shapes and ratios. And do not read a good `ece` next to a bad
`rmse` as good news: a model that hedges everything is perfectly calibrated and
useless.

## 5.7 What to hand in

* these two tables;
* at least three of the figures above, discussed rather than merely included;
* the derivation behind one thing you implemented, in your own notation: the
  Gaussian KL, the `1 / n_train` scaling, or the variance decomposition;
* your open-ended experiment: question, hypothesis, design, result, including when
  the result contradicts the hypothesis;
* a closing paragraph titled "what I would not trust this model for", written about
  your own track and your own numbers.

The figures are in `results/toy1d/`. Everything regenerates by running notebooks 01
to 05 in order.